## Middleware
### Middleware provides a way to more tightly control what happends inside the agent. Middleware is useful for the following
- tracking agent behaviour with logging, analytics and debugging
- transforming prompts, tool, selection, and output formatting.
- adding retries, fallbacks and early termination logic.
- apply applying rate limits, guardrails, and PII detection

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


### Summarization middleware

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware.summarization import SummarizationMiddleware

# Initialize the chat model (e.g., using OpenAI gpt-4o-mini or Groq llama-3.3-70b-versatile)
model = init_chat_model("gpt-4o-mini")

# Initialize SummarizationMiddleware
# trigger: triggers summarization when message history reaches 4 or more messages
# keep: retains only the 2 most recent messages (and summarizes the rest)
summarizer = SummarizationMiddleware(
    model=model,
    trigger=("messages", 10),
    keep=("messages", 2)
)

# Create the agent with the summarizer middleware
agent = create_agent(
    model=model,
    middleware=[summarizer]
)

# Test the agent with multiple conversation turns to trigger the summarizer
print("--- Turn 1 ---")
res1 = agent.invoke({"messages": [{"role": "user", "content": "Hi, my name is Uday."}]})
print("Agent:", res1["messages"][-1].content)

print("\n--- Turn 2 ---")
# Extend messages list manually
messages = res1["messages"] + [{"role": "user", "content": "I live in New York and my favorite color is blue."}]
res2 = agent.invoke({"messages": messages})
print("Agent:", res2["messages"][-1].content)

print("\n--- Turn 3 ---")
# Extend messages list manually (this will trigger the summarization middleware)
messages = res2["messages"] + [{"role": "user", "content": "What is my name, my favorite color, and where do I live?"}]
res3 = agent.invoke({"messages": messages})
print("Agent:", res3["messages"][-1].content)

print("\nFinal message history:")
for msg in res3["messages"]:
    print(f"[{msg.type}]: {msg.content}")


--- Turn 1 ---
Agent: Hi Uday! How can I assist you today?

--- Turn 2 ---
Agent: That's great, Uday! New York is an amazing place with so much to see and do, and blue is such a calming color. Do you have any favorite places in New York or activities you enjoy?

--- Turn 3 ---
Agent: Your name is Uday, your favorite color is blue, and you live in New York. If there's anything else you'd like to share or ask, feel free!

Final message history:
[human]: Hi, my name is Uday.
[ai]: Hi Uday! How can I assist you today?
[human]: I live in New York and my favorite color is blue.
[ai]: That's great, Uday! New York is an amazing place with so much to see and do, and blue is such a calming color. Do you have any favorite places in New York or activities you enjoy?
[human]: What is my name, my favorite color, and where do I live?
[ai]: Your name is Uday, your favorite color is blue, and you live in New York. If there's anything else you'd like to share or ask, feel free!


## Human in the loop


In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

def read_email_tool(email_id: str)-> str:
    """Mock function to read an email my its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str)-> str:
    """Mock function to send the email"""
    return f"email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="gpt-4o-mini",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["Approve","Edit","reject"]
                
                },
                "read_email_tool": False,
            }
        ),
    ],

)


In [6]:
from langchain_core.messages import HumanMessage
config= {"configurable": {"thread_id": "test-approve"}}

result= agent.invoke(
    {
        "messages": [HumanMessage(content="send email to uday@gmail.com with subject 'Hey' and body 'how are you'")]
    },
    config=config
)

In [9]:
# Approve step
from langgraph.types import Command

if "__interrupt__" in result:
    print(" Paused , Approving...")

    result=agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "Approve"}
                ]
            }
        ),
        config=config
    )

    print(f" Result: { result['messages'][-1].content}")
